In [1]:
import os
import pandas as pd
from sqlalchemy import create_engine
import yfinance as yf

# 1. Setup the connection using your environment variables
DB_USER = os.environ.get("DB_USER")
DB_PASSWORD = os.environ.get("DB_PASSWORD")
DB_HOST = os.environ.get("DB_HOST")
DB_PORT = os.environ.get("DB_PORT", "5432")
DB_NAME = os.environ.get("DB_NAME")

connection_string = f"postgresql+psycopg://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(connection_string)

class DataFetcher:
    def __init__(self, engine):
        self.engine = engine

    def fetch_data(self, symbol, limit=1000):
        # NOTE: Using 'new_data_ohlcv_1d' as per your friend's working code
        query = f"""
        SELECT *
        FROM futures_data.new_data_ohlcv_1d
        WHERE symbol = '{symbol}'
        ORDER BY "time" DESC
        LIMIT {limit};
        """
        df = pd.read_sql(query, self.engine)
        return df

    def fetch_macro_data(self, limit=1000):
        # Fetches ZT (2Y), ZF (5Y) and merges with VIX from Yahoo Finance
        q = f"""
        SELECT "time", symbol, close
        FROM futures_data.new_data_ohlcv_1d
        WHERE symbol IN ('ZT', 'ZF')
        ORDER BY "time" DESC
        LIMIT {limit};
        """
        fut = pd.read_sql(q, self.engine)
        fut["time"] = pd.to_datetime(fut["time"], utc=True)
        fut["date"] = fut["time"].dt.floor("D")
        fut = fut.sort_values("time").drop_duplicates(subset=["date", "symbol"], keep="last")

        fut_wide = (
            fut.pivot(index="date", columns="symbol", values="close")
            .reset_index()
            .rename(columns={"date": "time", "ZT": "treas_2y_close", "ZF": "treas_5y_close"})
        )

        vix_raw = yf.download("^VIX", period="max", interval="1d", auto_adjust=False)["Close"]
        if isinstance(vix_raw, pd.DataFrame):
            vix_raw = vix_raw.iloc[:, 0]

        vix = vix_raw.rename("vix_close").to_frame().reset_index()
        vix.columns = ['time', 'vix_close'] # Ensure naming consistency
        vix["time"] = pd.to_datetime(vix["time"], utc=True).dt.floor("D")

        df = fut_wide.merge(vix, on="time", how="left").sort_values("time").reset_index(drop=True)
        return df.dropna(subset=["treas_2y_close", "treas_5y_close", "vix_close"])

# 2. Execute and pull data
fetcher = DataFetcher(engine)

try:
    # Pulling Natural Gas (NG) as a test
    df_prices = fetcher.fetch_data('NG', limit=500)
    print("--- Futures Data (NG) ---")
    print(df_prices.head())

    # Pulling Macro Data
    df_macro = fetcher.fetch_macro_data(limit=1000)
    print("\n--- Macro Data (Treasuries + VIX) ---")
    print(df_macro.head())
    
    print(f"\nSuccessfully fetched {len(df_prices)} price rows and {len(df_macro)} macro rows.")

except Exception as e:
    print(f"Error: {e}")

--- Futures Data (NG) ---
                       time symbol  volume   open   high    low  close
0 2025-11-03 00:00:00+00:00     NG  164138  4.120  4.298  4.087  4.231
1 2025-11-02 00:00:00+00:00     NG    4758  4.160  4.200  4.101  4.119
2 2025-10-31 00:00:00+00:00     NG  166820  4.063  4.157  4.015  4.116
3 2025-10-30 00:00:00+00:00     NG  141145  3.804  4.087  3.787  4.062
4 2025-10-29 00:00:00+00:00     NG  110495  3.820  3.864  3.752  3.803


[*********************100%***********************]  1 of 1 completed


--- Macro Data (Treasuries + VIX) ---
                       time  treas_5y_close  treas_2y_close  vix_close
0 2024-03-26 00:00:00+00:00      106.968750      102.292969      13.24
1 2024-03-27 00:00:00+00:00      107.031250      102.281250      12.78
2 2024-03-28 00:00:00+00:00      106.960938      102.222656      13.01
4 2024-04-01 00:00:00+00:00      106.562500      102.089844      13.65
5 2024-04-02 00:00:00+00:00      106.460938      102.093750      14.61

Successfully fetched 500 price rows and 404 macro rows.


In [2]:
def fetch_macro_data(self, limit=1000):
    # Pulls ONLY from your internal database (ZT and ZF)
    q = f"""
    SELECT "time", symbol, close
    FROM futures_data.new_data_ohlcv_1d
    WHERE symbol IN ('ZT', 'ZF')
    ORDER BY "time" DESC
    LIMIT {limit};
    """
    fut = pd.read_sql(q, self.engine)

    fut["time"] = pd.to_datetime(fut["time"], utc=True)
    fut["date"] = fut["time"].dt.floor("D")

    # Keep the latest timestamp for each date/symbol pair
    fut = fut.sort_values("time").drop_duplicates(subset=["date", "symbol"], keep="last")

    # Pivot to wide format
    fut_wide = (
        fut.pivot(index="date", columns="symbol", values="close")
        .reset_index()
        .rename(columns={"date": "time", "ZT": "treas_2y_close", "ZF": "treas_5y_close"})
    )

    return fut_wide.sort_values("time").reset_index(drop=True)

In [9]:
import os
import pandas as pd
from algogators_data.algodata import Database

# 1. Initialize the Database connection
# Ensure your .env file has DB_NAME, DB_USER, DB_PASSWORD, and DB_HOST
db = Database(
    dbname=os.environ.get("DB_NAME"),
    username=os.environ.get("DB_USER"),
    password=os.environ.get("DB_PASSWORD"),
    host=os.environ.get("DB_HOST"),
)

print("--- Database Diagnostics ---")

# 2. Check which schemas exist (to find where the data is hidden)
db.cur.execute("SELECT schema_name FROM information_schema.schemata;")
schemas = [s[0] for s in db.cur.fetchall()]
print(f"Available Schemas: {schemas}")

# 3. Search for any table that contains 'ohlcv' to find the correct relation
db.cur.execute("""
    SELECT table_schema, table_name 
    FROM information_schema.tables 
    WHERE table_name LIKE '%ohlcv%';
""")
tables = db.cur.fetchall()
if tables:
    print("\nFound these OHLCV tables:")
    for schema, table in tables:
        print(f" - {schema}.{table}")
else:
    print("\nNo OHLCV tables found. Check your DB_NAME and permissions.")

# 4. Check what the library thinks are valid 'FT' (Futures) symbols
try:
    # This is the call that failed in your traceback
    symbols = db.get_list_symbols("FT")
    print(f"\nSuccess! Found {len(symbols)} symbols. First 5: {symbols[:5]}")
except Exception as e:
    print(f"\nCould not list symbols: {e}")

--- Database Diagnostics ---
Available Schemas: ['public', 'futures_data', 'information_schema', 'pg_catalog', 'pg_toast']

No OHLCV tables found. Check your DB_NAME and permissions.

Could not list symbols: relation "futures_data.ohlcv_1d" does not exist
LINE 2:             SELECT DISTINCT(symbol) FROM futures_data.ohlcv_...
                                                 ^



In [12]:
import os
from algogators_data.algodata import Database

# Initialize connection
db = Database(
    dbname=os.environ.get("DB_NAME"),
    username=os.environ.get("DB_USER"),
    password=os.environ.get("DB_PASSWORD"),
    host=os.environ.get("DB_HOST"),
)

# 1. List every table in the entire database to find where the data went
print("--- ALL VISIBLE TABLES ---")
db.cur.execute("""
    SELECT table_schema, table_name 
    FROM information_schema.tables 
    WHERE table_schema NOT IN ('information_schema', 'pg_catalog')
    ORDER BY table_schema, table_name;
""")
all_tables = db.cur.fetchall()

if all_tables:
    for schema, table in all_tables:
        print(f"{schema}.{table}")
else:
    print("No tables found. Your user may lack permissions or the DB is empty.")

# 2. Check if there are any tables at all inside the 'futures_data' schema
print("\n--- TABLES IN 'futures_data' ---")
db.cur.execute("""
    SELECT table_name 
    FROM information_schema.tables 
    WHERE table_schema = 'futures_data';
""")
futures_tables = db.cur.fetchall()
print([t[0] for t in futures_tables] if futures_tables else "None found.")

--- ALL VISIBLE TABLES ---
No tables found. Your user may lack permissions or the DB is empty.

--- TABLES IN 'futures_data' ---
None found.


In [1]:
import algogators_data.algodata as algodata

print("Available functions/objects in algodata:")
for item in dir(algodata):
    if not item.startswith('_'):
        print(f" - {item}")


Available functions/objects in algodata:
 - Database
 - SymbolNotFoundError
 - calendar
 - date
 - os
 - pd
 - pg
 - statistics
 - timedelta


In [2]:
import pkgutil
import algogators_data

print("Available submodules in algogators-data:")
for importer, modname, ispkg in pkgutil.iter_modules(algogators_data.__path__):
    print(f" - {modname}")


Available submodules in algogators-data:
 - algodata


In [3]:
import sys
!{sys.executable} -m pip install algogators-data python-dotenv



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip


In [4]:
"""Wasserstein risk index (distribution shift) for futures.

This notebook demonstrates a small end-to-end research pipeline:

- Load continuous daily futures prices (placeholder fake generator for now)
- Compute daily log returns
- Build the Wasserstein distribution-shift index W_t from cross-sectional returns
- Build a core panel with realized vol + rolling largest eigenvalue
- Run a simple HAC regression, an event study, and a conditioning experiment

TODO: Replace the fake loader in `algogators_wrisk.data.load_continuous_futures_prices`
with your internal `algogators-data` call.
"""

'Wasserstein risk index (distribution shift) for futures.\n\nThis notebook demonstrates a small end-to-end research pipeline:\n\n- Load continuous daily futures prices (placeholder fake generator for now)\n- Compute daily log returns\n- Build the Wasserstein distribution-shift index W_t from cross-sectional returns\n- Build a core panel with realized vol + rolling largest eigenvalue\n- Run a simple HAC regression, an event study, and a conditioning experiment\n\nTODO: Replace the fake loader in `algogators_wrisk.data.load_continuous_futures_prices`\nwith your internal `algogators-data` call.\n'

## Setup & imports

This notebook is a thin front-end: it calls into `algogators_wrisk` for all computations.

In [5]:
import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Make repo root importable when running from notebooks/
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from algogators_wrisk import config
from algogators_wrisk import data, features, analysis

plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.grid"] = True

np.random.seed(0)
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 50)

In [6]:
# Research parameters (override defaults here if desired)
universe = config.UNIVERSE
start_date = config.START_DATE
end_date = config.END_DATE

rv_past_window = config.RV_PAST_WINDOW
rv_future_window = config.RV_FUTURE_WINDOW
lambda1_window = config.LAMBDA1_WINDOW

w_event_q = config.W_EVENT_QUANTILE
exposure_on_event = config.EXPOSURE_ON_EVENT
hac_lags = config.HAC_LAGS

print(f"Universe size: {len(universe)}")
print(f"Date range: {start_date} → {end_date}")
print(f"RV windows (past/future): {rv_past_window}/{rv_future_window}")
print(f"Lambda1 window: {lambda1_window}")
print(f"Event quantile: {w_event_q}")
print(f"Exposure on event: {exposure_on_event}")
print(f"HAC lags: {hac_lags}")

Universe size: 12
Date range: 2015-01-01 → 2025-12-31
RV windows (past/future): 20/20
Lambda1 window: 60
Event quantile: 0.95
Exposure on event: 0.5
HAC lags: 5


## Load data

We use the placeholder loader in `algogators_wrisk.data` (fake, deterministic) so the notebook runs end-to-end.

(parameters are set in the code cell above)

## Compute returns and core panel

Compute log returns, build the cross-sectional return matrix, then build the core daily panel used for regression / event studies / strategy conditioning.

In [7]:
# 1) Load continuous futures prices (FAKE placeholder)
prices = data.load_continuous_futures_prices(
    universe=universe,
    start_date=start_date,
    end_date=end_date,
    seed=42,
)

display(prices.head())
print(prices.shape)

UndefinedTable: relation "futures_data.ohlcv_1d" does not exist
LINE 2:             SELECT DISTINCT(symbol) FROM futures_data.ohlcv_...
                                                 ^


In [ ]:
# 2) Compute daily log returns
returns = data.compute_log_returns(prices)
ret_matrix = features.build_return_matrix(returns, universe=universe)

display(ret_matrix.head())
print(ret_matrix.shape)

## Basic diagnostics

Plot the distribution-shift index \(W_t\), realized volatility, and the rolling largest correlation eigenvalue.

In [ ]:
# 3) Compute W_t, realized volatility, and rolling lambda1 via a single panel
panel = analysis.build_core_panel(
    ret_matrix,
    rv_past_window=rv_past_window,
    rv_future_window=rv_future_window,
    lambda1_window=lambda1_window,
    annualize_rv=True,
)

display(panel.tail())
print(panel.isna().mean().sort_values(ascending=False).head(10))

## Basic diagnostics (plots)

Plot \(W_t\), realized volatility, and \(\lambda_1\) over time.

In [ ]:
# Plot W_t, realized vol, and lambda1
fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)

panel["W"].plot(ax=axes[0], color="black", lw=1)
axes[0].set_title("Wasserstein distribution-shift index (W)")

panel[["rv_past", "rv_future"]].plot(ax=axes[1], lw=1)
axes[1].set_title("Realized volatility (past vs future)")

panel["lambda1"].plot(ax=axes[2], color="tab:purple", lw=1)
axes[2].set_title("Rolling largest correlation eigenvalue (lambda1)")

plt.tight_layout()
plt.show()

In [ ]:
# 5) Regression: rv_future ~ W + rv_past + lambda1 (HAC SEs)
res = analysis.run_rv_regression(panel, hac_lags=hac_lags)
print(res.summary())

## Event study

Event days are defined as high-\(W_t\) days (quantile threshold). We plot the average path of `mkt_ret` around events.

In [ ]:
es = analysis.make_event_study_dataset(
    panel,
    value_col="mkt_ret",
    quantile=w_event_q,
    pre=10,
    post=10,
    min_gap=5,
)

print(f"Identified {len(es.events)} event days (after de-clustering).")
display(es.stacked.head())

plt.figure(figsize=(10, 4))
es.avg_path.plot(marker="o")
plt.axvline(0, color="black", lw=1)
plt.title("Event study: average mkt_ret around high-W events")
plt.xlabel("tau (days from event)")
plt.ylabel("avg return")
plt.tight_layout()
plt.show()

## Strategy conditioning

Compare a baseline exposure strategy vs. scaling exposure down on high-\(W_t\) days. Compute basic stats: mean, stdev, Sharpe, max drawdown.

In [ ]:
# Baseline vs conditioned exposure (scaled down on high-W days)
strat = analysis.run_strategy_conditioning_experiment(
    panel,
    quantile=w_event_q,
    exposure_on_event=exposure_on_event,
)

display(strat.head())

# --- performance helpers ---
def sharpe_ratio(x: pd.Series, *, periods_per_year: int = 252) -> float:
    x = x.dropna()
    if x.std() == 0 or len(x) < 2:
        return np.nan
    return float(np.sqrt(periods_per_year) * x.mean() / x.std())


def max_drawdown(cum_pnl: pd.Series) -> float:
    """Max drawdown on cumulative PnL series (additive)."""
    c = cum_pnl.dropna()
    if c.empty:
        return np.nan
    running_max = c.cummax()
    dd = c - running_max
    return float(dd.min())


stats = pd.DataFrame(
    {
        "mean": [strat["baseline_pnl"].mean(), strat["conditioned_pnl"].mean()],
        "stdev": [strat["baseline_pnl"].std(), strat["conditioned_pnl"].std()],
        "sharpe": [
            sharpe_ratio(strat["baseline_pnl"]),
            sharpe_ratio(strat["conditioned_pnl"]),
        ],
        "max_dd": [
            max_drawdown(strat["baseline_cum"]),
            max_drawdown(strat["conditioned_cum"]),
        ],
    },
    index=["baseline", "conditioned"],
)

print(f"Event rate: {strat['is_event'].mean():.3%}")
display(stats)

plt.figure(figsize=(12, 4))
strat[["baseline_cum", "conditioned_cum"]].plot(lw=1.5)
plt.title("Cumulative PnL (baseline vs conditioned)")
plt.xlabel("date")
plt.ylabel("cum pnl (sum of daily returns)")
plt.tight_layout()
plt.show()

## Notes

- The price loader is currently **fake** and deterministic.
- Once you wire real prices, everything downstream should work unchanged.

In [ ]:
# Normalize the features
X_normalized = features.normalize_features(df_returns)
X_normalized_df = pd.DataFrame(
    X_normalized,
    columns=[f'{col}_norm' for col in df_returns.columns]
)

print("Normalized Features Statistics:")
print(X_normalized_df.describe())

# Scale features to [0, 1] range
X_scaled = features.scale_features(df_returns)
X_scaled_df = pd.DataFrame(
    X_scaled,
    columns=[f'{col}_scaled' for col in df_returns.columns]
)

print("\nScaled Features Statistics:")
print(X_scaled_df.describe())

In [ ]:
print("Configuration Settings:")
print(f"Data Path: {config.DATA_PATH}")
print(f"Output Path: {config.OUTPUT_PATH}")
print(f"Random Seed: {config.RANDOM_SEED}")
print(f"Test Size: {config.TEST_SIZE}")
print(f"Default Wasserstein P: {config.DEFAULT_WASSERSTEIN_P}")

# Set random seed for reproducibility
np.random.seed(config.RANDOM_SEED)